# License Plate Detection — Evaluation & Comparison
**CMPS 261 — Machine Learning Project**

Loads trained weights locally, runs inference on the test set, computes metrics, and generates comparison plots.

In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import os, json, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms.functional as TF

RESULTS_DIR  = '../results'
MODELS_DIR   = '../models'
TEST_IMG_DIR = '../data/yolo/images/test'
TEST_LBL_DIR = '../data/yolo/labels/test'
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: mps


## 1. Evaluate YOLOv8s on Test Set

In [2]:
from ultralytics import YOLO
import yaml

yolo_model = YOLO(os.path.join(MODELS_DIR, 'yolov8s_best.pt'))

DATA_YAML = os.path.abspath('../data/yolo/dataset.yaml')
DATA_DIR  = os.path.abspath('../data/yolo')

# Patch dataset.yaml so YOLO resolves image paths from the yaml's directory, not cwd
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATA_DIR
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f)

yolo_val = yolo_model.val(
    data=DATA_YAML,
    split='test',
    verbose=False
)

yolo_metrics = {
    'model'      : 'YOLOv8s',
    'precision'  : round(float(yolo_val.box.mp),    4),
    'recall'     : round(float(yolo_val.box.mr),    4),
    'map50'      : round(float(yolo_val.box.map50), 4),
    'map50_95'   : round(float(yolo_val.box.map),   4),
    'f1'         : round(2 * float(yolo_val.box.mp) * float(yolo_val.box.mr) /
                        (float(yolo_val.box.mp) + float(yolo_val.box.mr) + 1e-6), 4),
}

print(json.dumps(yolo_metrics, indent=2))

# Save
with open(os.path.join(RESULTS_DIR, 'yolo_metrics.json'), 'w') as f:
    json.dump(yolo_metrics, f, indent=2)

Ultralytics 8.4.37 🚀 Python-3.13.5 torch-2.11.0 CPU (Apple M4)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1034.6±83.6 MB/s, size: 570.9 KB)
val: Scanning /Users/ward/Desktop/aub/261/project/261_code/license plate/data/yolo/labels/test.cache... 66 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 10.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.4s/it 11.9s4.0s
                   all         66         71      0.918      0.915      0.947      0.518
Speed: 0.3ms preprocess, 172.6ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /Users/ward/Desktop/aub/261/project/261_code/license plate/notebooks/runs/detect/val9
{
  "model": "YOLOv8s",
  "precision": 0.9182,
  "recall": 0.9155,
  "map50": 0.9467,
  "map50_95": 0.5176,
  "f1": 0.9169
}


## 2. Evaluate Faster R-CNN on Test Set

In [3]:
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Dataset class (YOLO txt → Faster R-CNN format)
class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'):
                continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx - w/2) * W)
                ymin = max(0.0, (cy - h/2) * H)
                xmax = min(float(W), (cx + w/2) * W)
                ymax = min(float(H), (cy + h/2) * H)
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
        if not boxes:
            boxes = [[0.0, 0.0, 1.0, 1.0]]
        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return TF.to_tensor(img), {'boxes': boxes, 'labels': labels}

def collate_fn(batch):
    return tuple(zip(*batch))

# Load model
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
frcnn_model = fasterrcnn_resnet50_fpn_v2(weights=weights)
in_features = frcnn_model.roi_heads.box_predictor.cls_score.in_features
frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
frcnn_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'fasterrcnn_best.pth'), map_location=DEVICE))
frcnn_model.to(DEVICE).eval()

test_ds     = LicensePlateDataset(TEST_IMG_DIR, TEST_LBL_DIR)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
print(f'Test set: {len(test_ds)} images')

# Evaluate
def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0, xB-xA) * max(0, yB-yA)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)

THRESHOLD = 0.8
tp, fp, fn = 0, 0, 0
iou_scores = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(DEVICE) for img in images]
        preds  = frcnn_model(images)
        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= THRESHOLD].cpu().numpy()
            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched:
                    tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched)

precision = tp / (tp + fp + 1e-6)
recall    = tp / (tp + fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)
mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0

frcnn_metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN v2)',
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
print(json.dumps(frcnn_metrics, indent=2))

with open(os.path.join(RESULTS_DIR, 'fasterrcnn_metrics.json'), 'w') as f:
    json.dump(frcnn_metrics, f, indent=2)

Test set: 66 images
{
  "model": "Faster R-CNN (ResNet50-FPN v2)",
  "precision": 0.8732,
  "recall": 0.8732,
  "f1": 0.8732,
  "mean_iou": 0.7839
}


In [4]:
import ssl, certifi, urllib.request, os

url = 'https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth'
out = os.path.expanduser('~/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth')
os.makedirs(os.path.dirname(out), exist_ok=True)

if not os.path.exists(out):
    ssl_ctx = ssl.create_default_context(cafile=certifi.where())
    urllib.request.urlretrieve(url, out)
    print('Downloaded.')
else:
    print('Already cached.')

Already cached.


## 3. Evaluate RetinaNet on Test Set

In [5]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

# Load model
weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
retina_model = retinanet_resnet50_fpn_v2(weights=weights)
num_anchors  = retina_model.head.classification_head.num_anchors
in_channels  = retina_model.head.classification_head.conv[0][0].in_channels
retina_model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=2,
    norm_layer=torch.nn.BatchNorm2d,
)
retina_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'retinanet_best.pth'), map_location=DEVICE))
retina_model.to(DEVICE).eval()
print(f'Test set: {len(test_ds)} images')

RETINA_THRESHOLD = 0.45
tp, fp, fn = 0, 0, 0
iou_scores = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(DEVICE) for img in images]
        preds  = retina_model(images)
        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= RETINA_THRESHOLD].cpu().numpy()
            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched:
                    tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched)

precision = tp / (tp + fp + 1e-6)
recall    = tp / (tp + fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)
mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0

retina_metrics = {
    'model'    : 'RetinaNet (ResNet50-FPN v2)',
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
print(json.dumps(retina_metrics, indent=2))

with open(os.path.join(RESULTS_DIR, 'retinanet_metrics.json'), 'w') as f:
    json.dump(retina_metrics, f, indent=2)

Test set: 66 images
{
  "model": "RetinaNet (ResNet50-FPN v2)",
  "precision": 0.8667,
  "recall": 0.9155,
  "f1": 0.8904,
  "mean_iou": 0.7823
}


In [6]:
rows = [
    {
        'Model'       : 'YOLOv8s',
        'Precision'   : yolo_metrics['precision'],
        'Recall'      : yolo_metrics['recall'],
        'F1'          : yolo_metrics['f1'],
        'mAP@0.5'     : yolo_metrics['map50'],
        'mAP@0.5:0.95': yolo_metrics['map50_95'],
    },
    {
        'Model'       : 'RetinaNet',
        'Precision'   : retina_metrics['precision'],
        'Recall'      : retina_metrics['recall'],
        'F1'          : retina_metrics['f1'],
        'mAP@0.5'     : '-',
        'mAP@0.5:0.95': '-',
    },
    {
        'Model'       : 'Faster R-CNN',
        'Precision'   : frcnn_metrics['precision'],
        'Recall'      : frcnn_metrics['recall'],
        'F1'          : frcnn_metrics['f1'],
        'mAP@0.5'     : '-',
        'mAP@0.5:0.95': '-',
    },
]
df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df

              Precision  Recall      F1 mAP@0.5 mAP@0.5:0.95
Model                                                       
YOLOv8s          0.9182  0.9155  0.9169  0.9467       0.5176
RetinaNet        0.8667  0.9155  0.8904       -            -
Faster R-CNN     0.8732  0.8732  0.8732       -            -


,Precision,Recall,F1,mAP@0.5,mAP@0.5:0.95
Model,,,,,
YOLOv8s,0.9182,0.9155,0.9169,0.9467,0.5176
RetinaNet,0.8667,0.9155,0.8904,-,-
Faster R-CNN,0.8732,0.8732,0.8732,-,-


In [7]:
labels      = ['Precision', 'Recall', 'F1']
yolo_vals   = [yolo_metrics['precision'],   yolo_metrics['recall'],   yolo_metrics['f1']]
retina_vals = [retina_metrics['precision'], retina_metrics['recall'], retina_metrics['f1']]
frcnn_vals  = [frcnn_metrics['precision'],  frcnn_metrics['recall'],  frcnn_metrics['f1']]

x, width = np.arange(len(labels)), 0.25
fig, ax  = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width,  yolo_vals,   width, label='YOLOv8s',      color='#4C9BE8', alpha=0.85)
bars2 = ax.bar(x,          retina_vals, width, label='RetinaNet',     color='#6BCB77', alpha=0.85)
bars3 = ax.bar(x + width,  frcnn_vals,  width, label='Faster R-CNN',  color='#E87B4C', alpha=0.85)

for bar in list(bars1) + list(bars2) + list(bars3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — License Plate Detection')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Saved: results/model_comparison.png')

<Figure size 1000x500 with 1 Axes>

Saved: results/model_comparison.png


In [8]:
test_images = random.sample(os.listdir(TEST_IMG_DIR), 4)
fig, axes   = plt.subplots(4, 4, figsize=(18, 16))
CONF = 0.5

for row_idx, fname in enumerate(test_images):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    img_pil  = Image.open(img_path).convert('RGB')
    img_np   = np.array(img_pil)

    axes[row_idx][0].imshow(img_np)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')

    # YOLOv8s
    result = yolo_model.predict(img_path, conf=CONF, verbose=False)[0]
    axes[row_idx][1].imshow(img_np)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        axes[row_idx][1].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none'))
        axes[row_idx][1].text(x1, y1-4, f'{box.conf[0]:.2f}', color='lime', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][1].set_title('YOLOv8s', fontsize=9)
    axes[row_idx][1].axis('off')

    # RetinaNet
    img_tensor = TF.to_tensor(img_pil)
    pred = predict_tta(img_tensor)
    axes[row_idx][2].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][2].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#6BCB77', facecolor='none'))
        axes[row_idx][2].text(x1, y1-4, f'{score:.2f}', color='#6BCB77', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][2].set_title('RetinaNet', fontsize=9)
    axes[row_idx][2].axis('off')

    # Faster R-CNN
    tensor = img_tensor.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = frcnn_model(tensor)[0]
    axes[row_idx][3].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][3].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#FF6B6B', facecolor='none'))
        axes[row_idx][3].text(x1, y1-4, f'{score:.2f}', color='#FF6B6B', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][3].set_title('Faster R-CNN', fontsize=9)
    axes[row_idx][3].axis('off')

plt.suptitle('Side-by-Side: Original | YOLOv8s | RetinaNet | Faster R-CNN', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'side_by_side_comparison.png'), dpi=150)
plt.show()
print('Saved: results/side_by_side_comparison.png')

NameError: name 'predict_tta' is not defined

test_images = random.sample(os.listdir(TEST_IMG_DIR), 4)
fig, axes   = plt.subplots(4, 4, figsize=(18, 16))
CONF = 0.5

for row_idx, fname in enumerate(test_images):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    img_pil  = Image.open(img_path).convert('RGB')
    img_np   = np.array(img_pil)

    axes[row_idx][0].imshow(img_np)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')

    # YOLOv8s
    result = yolo_model.predict(img_path, conf=CONF, verbose=False)[0]
    axes[row_idx][1].imshow(img_np)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        axes[row_idx][1].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none'))
        axes[row_idx][1].text(x1, y1-4, f'{box.conf[0]:.2f}', color='lime', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][1].set_title('YOLOv8s', fontsize=9)
    axes[row_idx][1].axis('off')

    # RetinaNet
    tensor = TF.to_tensor(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = retina_model(tensor)[0]
    axes[row_idx][2].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][2].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#6BCB77', facecolor='none'))
        axes[row_idx][2].text(x1, y1-4, f'{score:.2f}', color='#6BCB77', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][2].set_title('RetinaNet', fontsize=9)
    axes[row_idx][2].axis('off')

    # Faster R-CNN
    with torch.no_grad():
        pred = frcnn_model(tensor)[0]
    axes[row_idx][3].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][3].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#FF6B6B', facecolor='none'))
        axes[row_idx][3].text(x1, y1-4, f'{score:.2f}', color='#FF6B6B', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][3].set_title('Faster R-CNN', fontsize=9)
    axes[row_idx][3].axis('off')

plt.suptitle('Side-by-Side: Original | YOLOv8s | RetinaNet | Faster R-CNN', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'side_by_side_comparison.png'), dpi=150)
plt.show()
print('Saved: results/side_by_side_comparison.png')